## PHASE - 2 Data Cleaning & Preparation Strategy

In [2]:
import pandas as pd
import numpy as np

# Load raw dataset
df = pd.read_csv("../data/raw/application_train.csv")

df.shape

(307511, 122)

In [3]:
df['DAYS_EMPLOYED'].describe()


count    307511.000000
mean      63815.045904
std      141275.766519
min      -17912.000000
25%       -2760.000000
50%       -1213.000000
75%        -289.000000
max      365243.000000
Name: DAYS_EMPLOYED, dtype: float64

In [4]:
# Handle Special Encoded Values

df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
df['DAYS_EMPLOYED'].describe()


count    252137.000000
mean      -2384.169325
std        2338.360162
min      -17912.000000
25%       -3175.000000
50%       -1648.000000
75%        -767.000000
max           0.000000
Name: DAYS_EMPLOYED, dtype: float64

In [5]:
# Remove Columns with Very High Missing %

missing_percent = df.isnull().mean() * 100

high_missing_cols = missing_percent[missing_percent > 50].index

len(high_missing_cols)

# drop them: 

df.drop(columns=high_missing_cols, inplace=True)

In [6]:
df.shape

(307511, 81)

In [7]:
df.isnull().sum().sum()

np.int64(1671440)

In [8]:
# Separate Categorical & Numerical

categorical_cols = df.select_dtypes(include=['object', 'string']).columns
numerical_cols = df.select_dtypes(include=[np.number]).columns


In [9]:
# Handle Missing Values (Proper Strategy)
# Numerical imputation
for col in numerical_cols:
    df[col] = df[col].fillna(df[col].median())

# Categorical imputation
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])
    
df.isnull().sum().sum()

np.int64(0)

In [10]:
# Encode Categorical Variables

df_encoded = pd.get_dummies(df, drop_first=True)
df_encoded.shape

(307511, 181)

In [11]:
# Separate Features & Target

X = df_encoded.drop('TARGET', axis=1)
y = df_encoded['TARGET']

In [12]:
# Train-Test Split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [13]:
# Create processed data folder if not exists
import os
os.makedirs("../data/processed", exist_ok=True)

# Save processed datasets
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)
y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)